<a href="https://colab.research.google.com/github/Sayak-coder/SIH_26051/blob/main/thermal_energy_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder , MinMaxScaler
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from xgboost import XGBRFClassifier

In [17]:
df=pd.read_csv('/content/ladakh_thermal_energy_hourly_dataset.csv')
df.head()

,latitude,longitude,hour,shelter_volume_m3,wall_material,wall_thickness_cm,glazing_ratio,insulation_r_value,ghi_w_m2,ambient_temp_c,thermal_mass_kj_k,thermal_energy_kwh
0,33.8745,77.1851,0,115.4,Stone,27.9,0.26,3.07,0.0,-6.835534,57426.76,0.0
1,33.8745,77.1851,1,115.4,Stone,27.9,0.26,3.07,0.0,-7.630127,57426.76,0.0
2,33.8745,77.1851,2,115.4,Stone,27.9,0.26,3.07,0.0,-8.129629,57426.76,0.0
3,33.8745,77.1851,3,115.4,Stone,27.9,0.26,3.07,0.0,-8.300000,57426.76,0.0
4,33.8745,77.1851,4,115.4,Stone,27.9,0.26,3.07,0.0,-8.129629,57426.76,0.0


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24000 entries, 0 to 23999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   latitude            24000 non-null  float64
 1   longitude           24000 non-null  float64
 2   hour                24000 non-null  int64  
 3   shelter_volume_m3   24000 non-null  float64
 4   wall_material       24000 non-null  object 
 5   wall_thickness_cm   24000 non-null  float64
 6   glazing_ratio       24000 non-null  float64
 7   insulation_r_value  24000 non-null  float64
 8   ghi_w_m2            24000 non-null  float64
 9   ambient_temp_c      24000 non-null  float64
 10  thermal_mass_kj_k   24000 non-null  float64
 11  thermal_energy_kwh  24000 non-null  float64
dtypes: float64(10), int64(1), object(1)
memory usage: 2.2+ MB


In [19]:
le=LabelEncoder()
df['wall_material']=le.fit_transform(df['wall_material'])

In [20]:
df['wall_material'].unique()

array([3, 1, 2, 0])

In [21]:
df.isnull().sum()

,0
latitude,0
longitude,0
hour,0
shelter_volume_m3,0
wall_material,0
wall_thickness_cm,0
glazing_ratio,0
insulation_r_value,0
ghi_w_m2,0
ambient_temp_c,0


In [22]:
X=df.drop('thermal_energy_kwh',axis=1)
y=df['thermal_energy_kwh']

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=45)

In [23]:
minmax_scale=MinMaxScaler()
X_train['ambient_temp_c']=minmax_scale.fit_transform(X_train[['ambient_temp_c']])
X_test['ambient_temp_c']=minmax_scale.fit_transform(X_test[['ambient_temp_c']])

In [24]:
from sklearn.metrics import r2_score
lr_model=LinearRegression()
lr_model.fit(X_train,y_train)
y_pred=lr_model.predict(X_test)
print(r2_score(y_test,y_pred))

0.7758350733455301


In [25]:
from xgboost import XGBRegressor

xgb_model=XGBRegressor()
xgb_model.fit(X_train,y_train)
y_pred=xgb_model.predict(X_test)
print(r2_score(y_test,y_pred))

0.9982467020378151


testing the model with user difened value


In [26]:
!pip install pvlib

In [27]:
# from pvlib.location import lookup_altitude, Location
# from datetime import datetime
# from zoneinfo import ZoneInfo

# # Get user inputs for building characteristics
# latitude_longitude_string = input('latitude,longitude: ')
# shelter_volume_m3 = float(input('Shelter Volume: '))
# wall_material = input('Wall Material: ')
# wall_thickness_cm = float(input('Wall Thickness: '))
# glazing_ratio = float(input('Glazing Ratio in %: '))
# insulation_r_value = float(input('Insulation Value: '))

# # Parse latitude and longitude
# latitude_str, longitude_str = latitude_longitude_string.split(',')
# latitude = float(latitude_str)
# longitude = float(longitude_str)

# # Automatically determine altitude
# altitude = lookup_altitude(latitude=latitude, longitude=longitude)

# # Get current time in IST for GHI calculation
# # current_time = datetime.now(ZoneInfo("Asia/Kolkata"))

# # times = pd.DatetimeIndex([current_time])
# today = date.today()
# times=pd.date_range(start=f'{today} 00:00', end=f'{today} 23:59', freq='1h', tz='Asia/Kolkata')

# # Create a pvlib Location object
# location = Location(latitude, longitude, tz='Asia/Kolkata', altitude=altitude)

# # Calculate Global Horizontal Irradiance (GHI) using a clear-sky model
# # This gives an estimate for the current time and location under clear-sky conditions.
# clearsky_data = location.get_clearsky(times)
# ghi_w_m2 = np.max(clearsky_data['ghi'].to_numpy())

# # Automatically determine ambient_temp_c and thermal_mass_kj_k from the dataset's mean
# # (as a proxy for 'machine calling the value itself' without external APIs or complex models)
# if(wall_material == 'Stone'):
#   Material_factor=1.5
# elif(wall_material == 'Rammed_Earth'):
#   Material_factor=1.3
# elif(wall_material == 'Mud_Brick'):
#   Material_factor=1.2
# elif(wall_material == 'Concrete'):
#   Material_factor=1.0

# if wall_material == 'Concrete':
#     noise = np.random.normal(0, 50)
# elif wall_material == 'Mud_Brick':
#     noise = np.random.normal(0, 500)
# elif wall_material == 'Rammed_Earth':
#     noise = np.random.normal(0, 300)
# elif wall_material == 'Stone':
#     noise = np.random.normal(0, 400)

# # ambient_temp_c = df['ambient_temp_c'].mean()
# ambient_temp_c = 8.0
# thermal_mass_kj_k = (shelter_volume_m3 * wall_thickness_cm * Material_factor * 12)+ noise

# input_data = pd.DataFrame({
#     'latitude': [latitude],
#     'longitude': [longitude],
#     'shelter_volume_m3': [shelter_volume_m3],
#     'wall_material': [wall_material],
#     'wall_thickness_cm': [wall_thickness_cm],
#     'glazing_ratio': [glazing_ratio],
#     'insulation_r_value': [insulation_r_value],
#     'ghi_w_m2': [ghi_w_m2],
#     'ambient_temp_c': [ambient_temp_c],
#     'thermal_mass_kj_k': [thermal_mass_kj_k]
# })

# # Apply LabelEncoder to 'wall_material'
# input_data['wall_material'] = le.transform(input_data['wall_material'])

# # Apply MinMaxScaler to 'ambient_temp_c'
# input_data['ambient_temp_c'] = minmax_scale.transform(input_data[['ambient_temp_c']])

# # Predict using the xgb_model
# predicted_energy = xgb_model.predict(input_data)
# print(f"Predicted Thermal Energy (kWh): {predicted_energy[0]:.2f}")

In [28]:
from datetime import date,datetime, timedelta
from zoneinfo import ZoneInfo
import ipywidgets as widgets
from IPython.display import display
from pvlib.location import lookup_altitude, Location

latitude_input = widgets.FloatText(description='Latitude:', value=33.8745)
longitude_input = widgets.FloatText(description='Longitude:', value=77.1851)
volume_input = widgets.FloatText(description='Shelter Volume (m³):', value=115.4)
material_input = widgets.Dropdown(
    options=['Concrete', 'Mud_Brick', 'Rammed_Earth', 'Stone'],
    description='Wall Material:'
)
thickness_input = widgets.FloatText(description='Wall Thickness (cm):', value=27.9)
glazing_input = widgets.FloatText(description='Glazing Ratio (%):', value=26.0)
insulation_input = widgets.FloatText(description='Insulation R-value:', value=3.07)
predict_button = widgets.Button(description='Predict', button_style='primary')
output = widgets.Output()

input_fields = widgets.VBox([
    latitude_input,
    longitude_input,
    volume_input,
    material_input,
    thickness_input,
    glazing_input,
    insulation_input,
    predict_button,
    output
])


def predict_thermal_energy(_):
    with output:
        output.clear_output()
        try:
            material = material_input.value
            glazing_fraction = glazing_input.value / 100.0
            encoded_material = le.transform([material])[0]

            ist_time = datetime.now(ZoneInfo("Asia/Kolkata"))
            current_time=ist_time.strftime("%H:%M")

            altitude = lookup_altitude(latitude=latitude_input.value, longitude=longitude_input.value)
            location = Location(latitude_input.value, longitude_input.value, tz='Asia/Kolkata', altitude=altitude)
            today = date.today()
            # Create a date range up to the current hour
            times=pd.date_range(start=f'{today} 00:00', end=f'{today} {current_time}', freq='1h', tz='Asia/Kolkata')

            clearsky_data = location.get_clearsky(times)
            ghi_values = clearsky_data['ghi'].to_numpy()

            # Filter out zero values and calculate mean GHI safely
            valid_ghi_values = ghi_values[ghi_values != 0]
            if len(valid_ghi_values) > 0:
                ghi_w_m2 = np.average(valid_ghi_values)
            else:
                ghi_w_m2 = 0 # Default to 0 if no valid GHI data

            # material factor lookup (fill in real values used in training)
            material_factors = {
                'Concrete': 1.0,
                'Mud_Brick': 0.6,
                'Rammed_Earth': 0.8,
                'Stone': 1.2,
            }
            material_factor = material_factors[material]

            thermal_mass_kj_k = (
                volume_input.value * thickness_input.value * material_factor * 12
            )+material_factor

            prediction_input = pd.DataFrame([{
                'latitude': latitude_input.value,
                'longitude': longitude_input.value,
                'shelter_volume_m3': volume_input.value,
                'wall_material': encoded_material,
                'wall_thickness_cm': thickness_input.value,
                'glazing_ratio': glazing_fraction,
                'insulation_r_value': insulation_input.value,
                'ghi_w_m2': ghi_w_m2,
                'ambient_temp_c': 3.0,
                'thermal_mass_kj_k': thermal_mass_kj_k,
            }], columns=X.columns)

            # Use the minmax_scale fitted during training (from cell cX-E-cUnM-Dj)
            prediction_input['ambient_temp_c'] = minmax_scale.transform(
                prediction_input[['ambient_temp_c']]
            )

            prediction = xgb_model.predict(prediction_input)[0]
            print(f'Predicted Thermal Energy: {prediction:.2f} kWh')
        except Exception as error:
            print(f'Prediction error: {error}')


predict_button.on_click(predict_thermal_energy)
display(input_fields)
predict_thermal_energy(None)

In [29]:
# from datetime import date, timedelta
# from pvlib.location import lookup_altitude, Location
# ghi_w_m2,j=0,0
# altitude=lookup_altitude(latitude=34.62753660427372,longitude=78.72376503514941)
# location = Location(34.62753660427372, 78.72376503514941, tz='Asia/Kolkata', altitude=altitude)
# today = date.today()
# yesterday = today - timedelta(days=1)
# times=pd.date_range(start=f'{yesterday} 00:00', end=f'{yesterday} 23:59', freq='1h', tz='Asia/Kolkata')
# clearsky_data = location.get_clearsky(times)
# for i in range(24):
#   if(clearsky_data['ghi'].to_numpy()[i]!=0):
#     ghi_w_m2+=clearsky_data['ghi'].to_numpy()[i]
#     j+=1
# ghi_w_m2=ghi_w_m2/j
# print(ghi_w_m2)